<span style="color: rgb(99, 100, 102); font-family: Roboto, sans-serif; background-color: rgb(255, 255, 255);"> Township is built upon farming and production puzzle cores, and casual order board games. The player is harvesting crops such as wheat, corn, carrot, potato, sugarcane, cocoa, tomato, rubber, silk, strawberries, rice and pepper. Assets are used to produce goods in factories to earn coins and experience points.

[Source](https://en.wikipedia.org/wiki/Township_(video_game))<span style="background-color: rgb(255, 255, 255);"><br></span>

See the first 1000 rows

In [31]:
SELECT TOP (1000) 
      [portfolio].[township].[items].[Id] as itemId
      , [portfolio].[township].[items].[name] as itemName
      , [portfolio].[township].[items].[productiontime] as productionTime
      , [portfolio].[township].[constraints].name as constraintName
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  ORDER BY productionTime
  , constraintName; 
GO

(38 rows affected)

Total execution time: 00:00:00.019

itemId,itemName,productionTime,constraintName
1,gold,0,none
2,wheat,2,field
23,cow feed,4,feed mill
14,bread,5,bakery
3,corn,5,field
24,chicken feed,8,feed mill
4,carrot,10,field
27,cream,11,dairy factory
15,cookies,15,bakery
25,sheep feed,16,feed mill


List the items, their constraints, the production time and their dependancies.

In [32]:
SELECT TOP (1000) 
    [portfolio].[township].[items].[Id],
    [portfolio].[township].[items].[name] as [itemName],
    [portfolio].[township].[constraints].[name] AS [constraintName],
    [portfolio].[township].[items].[productiontime] as [productionTime],
    parentDetails.name AS [parentName],
    [portfolio].[township].[dependancies].[items] AS [numberOfItems]
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  LEFT JOIN [portfolio].[township].[dependancies] ON [portfolio].[township].[items].[Id] = [portfolio].[township].[dependancies].[itemId]
  JOIN (SELECT * FROM [portfolio].[township].[items]) as parentDetails ON [portfolio].[township].[dependancies].[parentId] = parentDetails.Id
  WHERE [portfolio].[township].[dependancies].[parentId] IS NULL

UNION ALL

SELECT TOP (1000) 
    [portfolio].[township].[items].[Id],
    [portfolio].[township].[items].[name] as [itemName],
    [portfolio].[township].[constraints].[name] AS [constraintName],
    [portfolio].[township].[items].[productiontime] as [productionTime],
    parentDetails.name AS [parentName],
    [portfolio].[township].[dependancies].[items] AS [numberOfItems]
  FROM [portfolio].[township].[items]
  JOIN [portfolio].[township].[constraints] ON [portfolio].[township].[items].[constraintId] = [portfolio].[township].[constraints].[Id]
  LEFT JOIN [portfolio].[township].[dependancies] ON [portfolio].[township].[items].[Id] = [portfolio].[township].[dependancies].[itemId]
  JOIN (SELECT * FROM [portfolio].[township].[items]) as parentDetails ON [portfolio].[township].[dependancies].[parentId] = parentDetails.Id
  WHERE [portfolio].[township].[dependancies].[parentId] IS NOT NULL
  ORDER BY productionTime, itemName;
GO


(45 rows affected)

Total execution time: 00:00:00.018

Id,itemName,constraintName,productionTime,parentName,numberOfItems
2,wheat,field,2,gold,0
23,cow feed,feed mill,4,wheat,2
23,cow feed,feed mill,4,corn,1
14,bread,bakery,5,wheat,2
3,corn,field,5,gold,1
24,chicken feed,feed mill,8,wheat,2
24,chicken feed,feed mill,8,carrot,1
4,carrot,field,10,gold,2
27,cream,dairy factory,11,milk,1
15,cookies,bakery,15,wheat,2


list all the constraints and the dependent items ordered by production time.

In [33]:
SELECT TOP (1000) 
    [portfolio].[township].[constraints].[Id] as constriantId
    , [portfolio].[township].[constraints].[name] as constraintName
    ,[portfolio].[township].[items].[Id]
    ,[portfolio].[township].[items].[name] as itemName
    ,[portfolio].[township].[items].[productiontime] as productionTime
  FROM [portfolio].[township].[constraints]
  JOIN [portfolio].[township].[items] on [portfolio].[township].[constraints].[Id] = [portfolio].[township].[items].[constraintId]
  ORDER BY constriantId, productionTime, itemName;

(38 rows affected)

Total execution time: 00:00:00.043

constriantId,constraintName,Id,itemName,productionTime
1,none,1,gold,0
2,field,2,wheat,2
2,field,3,corn,5
2,field,4,carrot,10
2,field,5,sugarcane,20
2,field,6,cotton,30
2,field,7,strawberry,60
2,field,8,tomato,120
2,field,9,pine tree,180
2,field,10,potato,240


List all posssible product combinations per constraint under the productuion time limit.

In [40]:
DECLARE @constraintId INT;
DECLARE @productionTimeLimit INT = 60;

-- Declare a cursor to iterate through each constraintId
DECLARE constraint_cursor CURSOR FOR
SELECT DISTINCT constraintId
FROM portfolio.township.items
WHERE constraintId NOT IN (1);

-- Open the cursor
OPEN constraint_cursor;

-- Fetch the first constraintId
FETCH NEXT FROM constraint_cursor INTO @constraintId;

-- Loop through each constraintId
WHILE @@FETCH_STATUS = 0
BEGIN
    -- Print the constraintId (for demonstration purposes)
    PRINT 'Processing constraintId: ' + CAST(@constraintId AS VARCHAR);
    -- Run the RecursiveCTE query for the current constraintId
    WITH RecursiveCTE AS (
    SELECT
        [portfolio].[township].[items].[constraintId]
        ,CAST([portfolio].[township].[items].[Id] AS VARCHAR(MAX)) AS Combination
        ,CAST([portfolio].[township].[items].[name] AS VARCHAR(MAX)) AS [description]
        ,[portfolio].[township].[items].[productiontime] AS totalProductionTime
    FROM [portfolio].[township].[items]
    WHERE constraintId = @constraintId 
    AND productiontime <= @productionTimeLimit

    UNION ALL

    SELECT
        i.constraintId
        ,rc.Combination + ',' + CAST(i.[Id] AS VARCHAR(MAX))
        ,rc.[description] + ',' + CAST(i.[name] AS VARCHAR(MAX))
        ,rc.totalProductionTime + i.productiontime
    FROM RecursiveCTE rc
    JOIN [portfolio].[township].[items] i ON i.Id > CAST(SUBSTRING(rc.Combination, LEN(rc.Combination) - CHARINDEX(',', REVERSE(rc.Combination)) + 2, LEN(rc.Combination)) AS INT)
    WHERE
        i.constraintId = @constraintId 
    AND
        rc.totalProductionTime + i.[productiontime] <= @productionTimeLimit
    AND NOT EXISTS (
            SELECT 1
            FROM [portfolio].[township].[items] ni
            WHERE ni.Id = i.Id
            AND ',' + rc.Combination + ',' LIKE '%,' + CAST(ni.Id AS VARCHAR(MAX)) + ',%'
        )
    )
    SELECT
        constraintId
        ,c.name
        ,Combination
        ,[description]
        ,totalProductionTime
    FROM
        RecursiveCTE
    JOIN 
        [portfolio].[township].[constraints] c ON c.Id = RecursiveCTE.constraintId
    ORDER BY
        constraintId, totalProductionTime DESC;
    -- Fetch the next constraintId
    FETCH NEXT FROM constraint_cursor INTO @constraintId;
END
-- Close and deallocate the cursor
CLOSE constraint_cursor;
DEALLOCATE constraint_cursor;

Processing constraintId: 2

(68 rows affected)

Processing constraintId: 3

(13 rows affected)

Processing constraintId: 4

...

constraintId,name,Combination,description,totalProductionTime
7,sugar factory,"32,31","syrup,sugar",60
7,sugar factory,"31,32","sugar,syrup",60
7,sugar factory,32,syrup,40
7,sugar factory,31,sugar,20


constraintId,name,Combination,description,totalProductionTime
8,dairy factory,29,butter,54
8,dairy factory,"28,27","cheese,cream",38
8,dairy factory,"27,28","cream,cheese",38
8,dairy factory,28,cheese,27
8,dairy factory,27,cream,11


constraintId,name,Combination,description,totalProductionTime


constraintId,name,Combination,description,totalProductionTime
14,feed mill,"26,23,24,25","bee feed,cow feed,chicken feed,sheep feed",52
14,feed mill,"25,23,24,26","sheep feed,cow feed,chicken feed,bee feed",52
14,feed mill,"24,23,25,26","chicken feed,cow feed,sheep feed,bee feed",52
14,feed mill,"23,24,25,26","cow feed,chicken feed,sheep feed,bee feed",52
14,feed mill,"24,25,26","chicken feed,sheep feed,bee feed",48
14,feed mill,"25,24,26","sheep feed,chicken feed,bee feed",48
14,feed mill,"26,24,25","bee feed,chicken feed,sheep feed",48
14,feed mill,"26,23,25","bee feed,cow feed,sheep feed",44
14,feed mill,"25,23,26","sheep feed,cow feed,bee feed",44
14,feed mill,"23,25,26","cow feed,sheep feed,bee feed",44


constraintId,name,Combination,description,totalProductionTime
